In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case[0])

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/00000
0


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [7]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

In [8]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0' and case[2] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1' and case[2] == '0':
    cntrl_vars_init = [1]
elif case[3] == '0' and case[2] == '1':
    cntrl_vars_init = [2,4]
elif case[3] == '1' and case[2] == '1':
    cntrl_vars_init = [3,5]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [9]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_0_' + case + '.pickle'
final_file_1 = 'control_1_' + case + '.pickle'

In [10]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [11]:
i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [12]:
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [13]:
i_range_ = []

for i in i_range:
    if type(bestControl_init[i]) == type(None):
        i_range_.append(i)

i_range = np.array(i_range_)
        
print(i_range)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    #if prev_i != -1:
    #    control0 = bestControl_init[prev_i][:,:,n_pre-1:-n_post+1]
    
    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)
    
    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    prev_i = i

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  1 , total integrated cost =  1584.168404036671
RUN  2 , total integrated cost =  1492.4296382018115
RUN  3 , total integrated cost =  1369.5524896899317
RUN  4 , total integrated cost =  1288.574446344559
RUN  5 , total integrated cost =  1165.4408132464087
RUN  6 , total integrated cost =  1077.3475121674248
RUN  7 , total integrated cost =  923.2628026158374
RUN  8 , total integrated cost =  810.3062365640878
RUN  9 , total integrated cost =  538.7231328265532
RUN  10 , total integrated cost =  445.463103696963
RUN  11 , total integrated cost =  336.5989937542213
RUN  12 , total integrated cost =  280.88166462062117
RUN  13 , total integrated cost =  219.35495024289997
RUN  14 , total integrated cost =  181.84188560243143
RUN  15 , total integrated cost =  87.41347566767857
RUN  1

ERROR:root:Problem in initial value trasfer


RUN  800 , total integrated cost =  33.34063824926766
Control only changes marginally.
RUN  805 , total integrated cost =  33.34063824926757
Improved over  805  iterations in  26.557921297848225  seconds by  99.34591440995139  percent.
Problem in initial value trasfer:  Vmean_exc -56.62446622072298 -56.624466266907824
weight =  1528.8519044207862
set cost params:  1.0 0.0 1528.8519044207862
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5095.003906676567
Gradient descend method:  None
RUN  1 , total integrated cost =  5064.815940056542
RUN  2 , total integrated cost =  5061.723685444849
RUN  3 , total integrated cost =  5055.082915074557
RUN  4 , total integrated cost =  5053.624637323703
RUN  5 , total integrated cost =  5051.259205589486
RUN  6 , total integrated cost =  5045.6848660965115
RUN  7 , total integrated cost =  5044.974885446075
RUN  8 , total integrated cost =  5044.085644841913
RUN  9 , total integrated cost =  5039.583534010494
RUN  10 , total 

ERROR:root:Problem in initial value trasfer


RUN  50 , total integrated cost =  5000.252892532592
Control only changes marginally.
RUN  50 , total integrated cost =  5000.252892532592
Improved over  50  iterations in  1.029802406206727  seconds by  1.8596848182944683  percent.
Problem in initial value trasfer:  Vmean_exc -56.62452557722461 -56.62452344281721
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13018.074640346456
Gradient descend method:  None
RUN  1 , total integrated cost =  3408.925269495024
RUN  2 , total integrated cost =  2119.0873436689185
RUN  3 , total integrated cost =  1578.140801886103
RUN  4 , total integrated cost =  1155.1751254259466
RUN  5 , total integrated cost =  934.8118821527049
RUN  6 , total integrated cost =  727.5215106147433
RUN  7 , total integrated cost =  608.6830638892762
RUN  8 , total integrated cost =  492.9691572106643
RUN  9 , total integrated cost =  419.58521356922495
RUN  10 , 

...

RUN  16 , total integrated cost =  33670.05009478856
RUN  17 , total integrated cost =  33670.05009478856
Control only changes marginally.
RUN  17 , total integrated cost =  33670.05009478856
Improved over  17  iterations in  0.44336634688079357  seconds by  2.3714282771085493  percent.
Problem in initial value trasfer:  Vmean_exc -56.70312025110278 -56.70312018684001
-------  85 0.47500000000000014 0.7250000000000004
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15143.755110304457
Gradient descend method:  None
RUN  1 , total integrated cost =  15143.755110304457
Control only changes marginally.
RUN  1 , total integrated cost =  15143.755110304457
Improved over  1  iterations in  0.053702253848314285  seconds by  0.0  percent.
weight =  10.0
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15143.755110304457
Gradient descend method:  None
RUN  1 , total integrated cost =  151

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  1244 , total integrated cost =  27.153882518042533
Improved over  1244  iterations in  25.325867723673582  seconds by  99.91843244055943  percent.
Problem in initial value trasfer:  Vmean_exc -56.703541478563814 -56.70354152642115
weight =  12259.775906726405
set cost params:  1.0 0.0 12259.775906726405
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33285.276799806205
Gradient descend method:  None
RUN  1 , total integrated cost =  32956.81767522253
RUN  2 , total integrated cost =  32955.998343733125
RUN  3 , total integrated cost =  32955.88495389538
RUN  4 , total integrated cost =  32955.826441069774
RUN  5 , total integrated cost =  32955.69083851573
RUN  6 , total integrated cost =  32955.14646257371
RUN  7 , total integrated cost =  32879.39726579287
RUN  8 , total integrated cost =  32841.224118180326


ERROR:root:Problem in initial value trasfer


RUN  9 , total integrated cost =  32841.202184722526
RUN  10 , total integrated cost =  32841.19728567421
RUN  11 , total integrated cost =  32841.196993222766
RUN  12 , total integrated cost =  32841.19694684844
RUN  13 , total integrated cost =  32841.19694317372
RUN  14 , total integrated cost =  32841.196943173716
RUN  15 , total integrated cost =  32841.1969431737
RUN  16 , total integrated cost =  32841.1969431737
Control only changes marginally.
RUN  16 , total integrated cost =  32841.1969431737
Improved over  16  iterations in  0.3887319825589657  seconds by  1.334163027405225  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354268101461 -56.703542670331494


In [15]:
#plot initial guesses
"""
for i in i_range:
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range:\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [16]:
found_solution = []
no_solution = []
last_update = -1
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]
i_range = range(0, len(exc),i_stepsize)
i_range_0 = []
i_range_1 = []

print(already_tried, len(already_tried))

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break
    
    #if last_update != k-1:
    #    print("no improvement from previous step")
    #    break

    for i in i_range:
        print("------- ", i, exc[i], inh[i])

        if np.abs(bestState_init[i][0,0,-1] - target[i][0,0,-1]) < 0.5 * np.abs(
            bestState_init[i][0,0,-1] - bestState_init[i][0,0,0]):
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
                #last_update = k
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        #if i not in no_solution:
        #    print("no solution for ", i)
        #    no_solution.append(i)
        
        if i not in i_range_0:
            i_range_0.append(i)
            
        if i not in i_range_1:
            i_range_1.append(i)

        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

[[], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], []] 147
------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
------- 

ERROR:root:Problem in initial value trasfer


RUN  200 , total integrated cost =  62.139560117408436
Control only changes marginally.
RUN  200 , total integrated cost =  62.139560117408436
Improved over  200  iterations in  4.097936309874058  seconds by  99.59073289597605  percent.
Problem in initial value trasfer:  Vmean_exc -56.67996219088091 -56.67996199326329
weight =  2437.0554090970986
set cost params:  1.0 0.0 2437.0554090970986
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  15143.090845775327
Gradient descend method:  None
RUN  1 , total integrated cost =  15133.57066779652
RUN  2 , total integrated cost =  15133.52320936234
RUN  3 , total integrated cost =  15133.522061037005
RUN  4 , total integrated cost =  15133.521952263614
RUN  5 , total integrated cost =  15133.521941696179
RUN  6 , total integrated cost =  15133.521940225999
RUN  7 , total integrated cost =  15133.521940190307


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  15133.521940190301
RUN  9 , total integrated cost =  15133.52194019029
RUN  10 , total integrated cost =  15133.521940190289
RUN  11 , total integrated cost =  15133.521940190289
Control only changes marginally.
RUN  11 , total integrated cost =  15133.521940190289
Improved over  11  iterations in  0.27391683869063854  seconds by  0.06318991071566415  percent.
Problem in initial value trasfer:  Vmean_exc -56.679915597417455 -56.67991657801176
-------  90 0.6000000000000003 0.7250000000000004
found solution for  90
-------  95 0.5250000000000001 0.7500000000000004
[0, 5, 10, 15, 20, 30, 35, 40, 45, 50, 55, 60, 70, 75, 80, 90] []
closest index  80
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  24167.4284154468
Gradient descend method:  None
RUN  1 , total integrated cost =  24132.88492746524
RUN  2 , total integrated cost =  24128.523889449265
RUN  3 , total integrated cost =  

...

Control only changes marginally.
RUN  424 , total integrated cost =  55.98187538022989
Improved over  424  iterations in  8.61276383139193  seconds by  84.35347365114873  percent.
Problem in initial value trasfer:  Vmean_exc -56.6398022167336 -56.6398022817082
weight =  1470.459352345179
set cost params:  1.0 0.0 1470.459352345179
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8230.588878113513
Gradient descend method:  None
RUN  1 , total integrated cost =  8212.086400823006
RUN  2 , total integrated cost =  8211.952700124371
RUN  3 , total integrated cost =  8211.850983008404
RUN  4 , total integrated cost =  8202.886604650657
RUN  5 , total integrated cost =  8198.382489605703
RUN  6 , total integrated cost =  8198.34276604285
RUN  7 , total integrated cost =  8198.341873116999


ERROR:root:Problem in initial value trasfer


RUN  8 , total integrated cost =  8198.34183209014
RUN  9 , total integrated cost =  8198.34183041501
RUN  10 , total integrated cost =  8198.34183033174
RUN  11 , total integrated cost =  8198.341830328573
RUN  12 , total integrated cost =  8198.341830328438
RUN  13 , total integrated cost =  8198.341830328436
RUN  14 , total integrated cost =  8198.34183032843
RUN  15 , total integrated cost =  8198.34183032843
Control only changes marginally.
RUN  15 , total integrated cost =  8198.34183032843
Improved over  15  iterations in  0.3675021678209305  seconds by  0.391795146892008  percent.
Problem in initial value trasfer:  Vmean_exc -56.64048562359129 -56.640468722928354
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.55000

ERROR:root:Problem in initial value trasfer


RUN  1700 , total integrated cost =  48.103759067058526
Control only changes marginally.
RUN  1701 , total integrated cost =  48.103759067058526
Improved over  1701  iterations in  34.70474381186068  seconds by  99.76033340053783  percent.
Problem in initial value trasfer:  Vmean_exc -56.69518489941081 -56.695184838333816
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
found solution for  85
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.7500000000000004
found solution for  95
-------  100 0.4500000000000001 0.7750000000000005
-------  105 0.5750000000000002 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
found solution for  115
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.850000000000000

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    if counter > 100:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  1557.5214245595384
set cost params:  1.0 0.0 1557.5214245595384
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5093.262231912901
Gradient descend method:  None
RUN  1 , total integrated cost =  5093.258380055736
RUN  2 , total integrated cost =  5093.258154823901
RUN  3 , total integrated cost =  5093.258144572473
RUN  4 , total integrated cost =  509

ERROR:root:Problem in initial value trasfer


RUN  7 , total integrated cost =  5093.258143920157
RUN  8 , total integrated cost =  5093.258143920147
RUN  9 , total integrated cost =  5093.258143920146
RUN  10 , total integrated cost =  5093.258143920144
RUN  11 , total integrated cost =  5093.258143920142
RUN  12 , total integrated cost =  5093.258143920141
RUN  13 , total integrated cost =  5093.258143920139
RUN  14 , total integrated cost =  5093.258143920139
Control only changes marginally.
RUN  14 , total integrated cost =  5093.258143920139
Improved over  14  iterations in  0.3718093652278185  seconds by  8.026275844486008e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.62452281169975 -56.624520708497414
-------  10 0.4250000000000001 0.42500000000000016
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  4116.300889142679
set cost params:  1.0 0.0 4116.300889142679
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13013.69345641113
Gradient descen

ERROR:root:Problem in initial value trasfer


RUN  19 , total integrated cost =  13013.687761017498
Control only changes marginally.
RUN  19 , total integrated cost =  13013.687761017498
Improved over  19  iterations in  0.44685989804565907  seconds by  4.376462111110868e-05  percent.
Problem in initial value trasfer:  Vmean_exc -56.67066142048298 -56.67066025477499
-------  20 0.4500000000000001 0.4750000000000002
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  1475.4796603950235
set cost params:  1.0 0.0 1475.4796603950235
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8226.27605060381
Gradient descend method:  None
RUN  1 , total integrated cost =  8226.276019558862
RUN  2 , total integrated cost =  8226.276017477625
RUN  3 , total integrated cost =  8226.276017356206
RUN  4 , total integrated cost =  8226.276017347542
RUN  5 , total integrated cost =  8226.276017346918
RUN  6 , total integrated cost =  8226.27601734688


...

RUN  1 , total integrated cost =  23527.807537859262
Control only changes marginally.
RUN  1 , total integrated cost =  23527.807537859262
Improved over  1  iterations in  0.08250032551586628  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70074459252311 -56.70074142158406
-------  140 0.4500000000000001 0.9000000000000006
converged for  140
-------  145 0.5750000000000002 0.9000000000000006
converged for  145
[[True, True], [False, False], [True, True], [False, False], [True, True], [True, False], [True, True], [True, True], [True, True], [True, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [False, False], [True, True], [True, False]]
-------  0 0.4000000000000001 0.3500000000000001
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
conver

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  14667.666412967443
RUN  2 , total integrated cost =  14667.66641296744
RUN  3 , total integrated cost =  14667.666412967435
RUN  4 , total integrated cost =  14667.666412967435
Control only changes marginally.
RUN  4 , total integrated cost =  14667.666412967435
Improved over  4  iterations in  0.23873591050505638  seconds by  7.105427357601002e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.70159251171578 -56.70154212425129
-------  100 0.4500000000000001 0.7750000000000005
converged for  100
-------  105 0.5750000000000002 0.7750000000000005
converged for  105
-------  110 0.5000000000000002 0.8000000000000005
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  120 0.5500000000000003 0.8250000000000005
converged for  120
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  130 0.6000000000000003 0.8500000000000005
converged for  130
-------  135 0.5250000000000001

ERROR:root:Problem in initial value trasfer


 45 0.5000000000000002 0.5750000000000003
converged for  45
-------  50 0.47500000000000014 0.6000000000000003
converged for  50
-------  55 0.4250000000000001 0.6250000000000003
converged for  55
-------  60 0.5500000000000003 0.6250000000000003
converged for  60
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  70 0.4500000000000001 0.6750000000000004
converged for  70
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  80 0.5250000000000001 0.7000000000000004
converged for  80
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  90 0.6000000000000003 0.7250000000000004
converged for  90
-------  95 0.5250000000000001 0.7500000000000004
no convergence
weight =  460.5405410160461
set cost params:  1.0 0.0 460.5405410160461
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  17800.75631530893
Gradient descend method:  None
RUN  1 , total integrated cost =  17800.75631530893
Control only cha

In [18]:
print(conv_init[::i_stepsize])

with open(init_file,'wb') as f:
    pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [19]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [20]:
i_range_0 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_0:
    if type(bestControl_0[i]) == type(None):
        i_range_.append(i)

i_range_0 = np.array(i_range_)
        
print(i_range_0)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [21]:
factor_iteration = 20
    
for i in i_range_0:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

# exc and inh control current 

    setinit(initVars[i], aln)
    aln.params.duration = dur
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
    #control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
 
    cost.setParams(1.0, 0.0, 10.0)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = "HS"
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_0[i][-j] == 0.:
        j += 1
    
    weight_ = 100 * cost_uncontrolled[i] / cost_0[i][-j] - 1
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 50 * factor_iteration

    weights_0[i] = cost.getParams()

    bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  73.52828700475851
Gradient descend method:  None
RUN  1 , total integrated cost =  50.751375522106486
RUN  2 , total integrated cost =  46.171924247958955
RUN  3 , total integrated cost =  42.65133480781874
RUN  4 , total integrated cost =  41.20822478611893
RUN  5 , total integrated cost =  39.7978419306499
RUN  6 , total integrated cost =  39.32682021760325
RUN  7 , total integrated cost =  38.72996151645165
RUN  8 , total integrated cost =  38.48577548238202
RUN  9 , total integrated cost =  38.05050898079225
RUN  10 , total integrated cost =  37.806732777617846
RUN  11 , total integrated cost =  37.15069401975447
RUN  12 , total integrated cost =  36.74056200350187
RUN  13 , total integrated cost =  35.89051470204409
RUN  14 , total integrated cost =  35.390610373178546
RUN  15 , total integrated cost =  34.92961499939255
RUN  16 ,

ERROR:root:Problem in initial value trasfer


RUN  80 , total integrated cost =  322.9340793265805
Control only changes marginally.
RUN  80 , total integrated cost =  322.9340793265805
Improved over  80  iterations in  5.829866975545883  seconds by  1.7101218204736313  percent.
Problem in initial value trasfer:  Vmean_exc -56.62447121632783 -56.624471126871136
weight =  1577.4304458758834
set cost params:  1.0 0.0 1577.4304458758834
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5087.5963141041275
Gradient descend method:  None
RUN  1 , total integrated cost =  5070.475384889866
RUN  2 , total integrated cost =  5053.481075115216
RUN  3 , total integrated cost =  5041.220220179993
RUN  4 , total integrated cost =  5041.171740197607
RUN  5 , total integrated cost =  5040.189544034278
RUN  6 , total integrated cost =  5039.411604176061
RUN  7 , total integrated cost =  5038.682915073461
RUN  8 , total integrated cost =  5037.92540315138
RUN  9 , total integrated cost =  5037.259293738515
RUN  10 , total inte

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  238 , total integrated cost =  3880.9702356882526
Improved over  238  iterations in  13.83847664669156  seconds by  23.717016915646326  percent.
Problem in initial value trasfer:  Vmean_exc -56.62562081551924 -56.62560077256377
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  113.0822851150616
Gradient descend method:  None
RUN  1 , total integrated cost =  34.43864201596662
RUN  2 , total integrated cost =  33.873491605024796
RUN  3 , total integrated cost =  33.384607108502415
RUN  4 , total integrated cost =  33.186024387763915
RUN  5 , total integrated cost =  33.00845967704251
RUN  6 , total integrated cost =  32.91841504396275
RUN  7 , total integrated cost =  32.81481827881805
RUN  8 , total integrated cost =  32.75910571185711
RUN  9 , total integrated cost =  32.68048036808627
RUN  10 , total integrated cost =  32.630691935673774
RUN  11

...

RUN  13 , total integrated cost =  23409.713618417536
Control only changes marginally.
RUN  13 , total integrated cost =  23409.713618417536
Improved over  13  iterations in  1.7191668190062046  seconds by  0.44404418299600934  percent.
Problem in initial value trasfer:  Vmean_exc -56.700658551045215 -56.70065944405976
-------  145 0.5750000000000002 0.9000000000000006
set cost params:  1.0 0.0 10.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  187.8777243111043
Gradient descend method:  None
RUN  1 , total integrated cost =  27.203172422928173
RUN  2 , total integrated cost =  27.15797552148688
RUN  3 , total integrated cost =  27.1510118869564
RUN  4 , total integrated cost =  27.142951141328417
RUN  5 , total integrated cost =  27.140629051612514
RUN  6 , total integrated cost =  27.135916803428525
RUN  7 , total integrated cost =  27.133314389034567
RUN  8 , total integrated cost =  27.115981587849706
RUN  9 , total integrated cost =  27.10760510149308
RUN

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  25 , total integrated cost =  268.6512478307947
Improved over  25  iterations in  3.0938656218349934  seconds by  0.7848927145310824  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354799822416 -56.70354751168142
weight =  12390.549168550624
set cost params:  1.0 0.0 12390.549168550624
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  33270.49984027785
Gradient descend method:  None
RUN  1 , total integrated cost =  33007.108065689994
RUN  2 , total integrated cost =  33006.485246840835
RUN  3 , total integrated cost =  33006.465991605866
RUN  4 , total integrated cost =  33006.46265529145
RUN  5 , total integrated cost =  33006.46244095826
RUN  6 , total integrated cost =  33006.46239220149
RUN  7 , total integrated cost =  33006.46238505533
RUN  8 , total integrated cost =  33006.46238410643
RUN  9 , total integrated cost =  33006.462383977414
RUN  10 , total integrated cost =  33006.46237971902
RUN  11 , total 

ERROR:root:Problem in initial value trasfer


RUN  12 , total integrated cost =  33006.46237448187
RUN  13 , total integrated cost =  33006.46237448187
Control only changes marginally.
RUN  13 , total integrated cost =  33006.46237448187
Improved over  13  iterations in  1.748118992894888  seconds by  0.7936083529359337  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354909867723 -56.70354856717717


In [22]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [23]:
factor_iteration = 20
full_converge = False
conv_0 = [[False]*2] * len(exc)

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 100:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_0:

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                       + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = 500 * factor_iteration

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    counter += 1
    

--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  2070.806707641438
set cost params:  1.0 0.0 2070.806707641438
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5074.65570087097
Gradient descend method:  None
RUN  1 , total integrated cost =  5073.5627758261635
RUN  2 , total integrated cost =  5073.561257394055
RUN  3 , total integrated cost =  5073.561255977022
RUN  4 , total integrated cost =  5073.561255977012
RUN  5 , total integrated cost =  5073.561255977011


ERROR:root:Problem in initial value trasfer


RUN  6 , total integrated cost =  5073.561255977011
Control only changes marginally.
RUN  6 , total integrated cost =  5073.561255977011
Improved over  6  iterations in  0.8237963300198317  seconds by  0.021566879774155723  percent.
Problem in initial value trasfer:  Vmean_exc -56.6259476163334 -56.62592529791007
-------  15 0.4500000000000001 0.4500000000000002
no convergence
weight =  4157.251601880289
set cost params:  1.0 0.0 4157.251601880289
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  13014.430955704365
Gradient descend method:  None
RUN  1 , total integrated cost =  13014.430244062212
RUN  2 , total integrated cost =  13014.430024512205
RUN  3 , total integrated cost =  13014.429967494683
RUN  4 , total integrated cost =  13014.429952283872
RUN  5 , total integrated cost =  13014.429947987544
RUN  6 , total integrated cost =  13014.429947096038
RUN  7 , total integrated cost =  13014.429946807471
RUN  8 , total integrated cost =  13014.429946726761
RU

ERROR:root:Problem in initial value trasfer


RUN  18 , total integrated cost =  13014.429946691245
Control only changes marginally.
RUN  18 , total integrated cost =  13014.429946691245
Improved over  18  iterations in  2.099160023033619  seconds by  7.753032946311578e-06  percent.
Problem in initial value trasfer:  Vmean_exc -56.670535033921276 -56.67053642302118
-------  25 0.4250000000000001 0.5000000000000002
no convergence
weight =  1475.7602355023935
set cost params:  1.0 0.0 1475.7602355023935
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8226.300740399825
Gradient descend method:  None
RUN  1 , total integrated cost =  8226.300724243874
RUN  2 , total integrated cost =  8226.300723734617
RUN  3 , total integrated cost =  8226.300723624245
RUN  4 , total integrated cost =  8226.300723603994
RUN  5 , total integrated cost =  8226.300723600003
RUN  6 , total integrated cost =  8226.300723599159
RUN  7 , total integrated cost =  8226.300723598983
RUN  8 , total integrated cost =  8226.300723598939
RU

...

RUN  1 , total integrated cost =  33287.387713639066
Control only changes marginally.
RUN  1 , total integrated cost =  33287.387713639066
Improved over  1  iterations in  0.20163679867982864  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354910620716 -56.70354857447866
--------------- 4
[[True, True], [False, False], [True, True], [True, False], [True, True], [True, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
weight =  2079.6260703708454
set cost params:  1.0 0.0 2079.6260703708454
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5094.839944353002
Gradient descend method

ERROR:root:Problem in initial value trasfer


RUN  2 , total integrated cost =  5094.839944353
RUN  3 , total integrated cost =  5094.839944353
Control only changes marginally.
RUN  3 , total integrated cost =  5094.839944353
Improved over  3  iterations in  0.5473072938621044  seconds by  2.842170943040401e-14  percent.
Problem in initial value trasfer:  Vmean_exc -56.6259562474754 -56.62593386944369
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125
-------  135 0.52500000

ERROR:root:Problem in initial value trasfer


RUN  0 , total integrated cost =  5094.839945476569
Gradient descend method:  None
RUN  1 , total integrated cost =  5094.839945476569
Control only changes marginally.
RUN  1 , total integrated cost =  5094.839945476569
Improved over  1  iterations in  0.19497697986662388  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.6259562474754 -56.62593386944369
-------  15 0.4500000000000001 0.4500000000000002
converged for  15
-------  25 0.4250000000000001 0.5000000000000002
converged for  25
-------  45 0.5000000000000002 0.5750000000000003
converged for  45
-------  65 0.5000000000000002 0.6500000000000004
converged for  65
-------  75 0.5750000000000002 0.6750000000000004
converged for  75
-------  85 0.47500000000000014 0.7250000000000004
converged for  85
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
converged for  115
-------  125 0.47500000000000014 0.8500000000000005
converged for  125


In [24]:
print(conv_0[::i_stepsize])

with open(final_file,'wb') as f:
    pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                 costnode_0, weights_0], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]


In [25]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [26]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [27]:
i_range_1 = range(0, len(exc),i_stepsize)
i_range_ = []

for i in i_range_1:
    if type(bestControl_1[i]) == type(None):
        i_range_.append(i)

i_range_1 = np.array(i_range_)  

print(i_range_1)

[  5  15  25  45  65  75  85  95 115 125 135 145]


In [28]:
factor_iteration = 20

for i in i_range_1:        

    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int( 500 * factor_iteration )

    weights_1[i] = cost.getParams()

    bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)

-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  79.8669975297144
Gradient descend method:  None
RUN  1 , total integrated cost =  3.507547796692944
RUN  2 , total integrated cost =  2.7055401337628493
RUN  3 , total integrated cost =  2.6282224874885523
RUN  4 , total integrated cost =  2.590845068162186
RUN  5 , total integrated cost =  2.5728096149450317
RUN  6 , total integrated cost =  2.5679260082159345
RUN  7 , total integrated cost =  2.564031043875492
RUN  8 , total integrated cost =  2.5621962098264093
RUN  9 , total integrated cost =  2.5604537467514175
RUN  10 , total integrated cost =  2.5593811739387173
RUN  11 , total integrated cost =  2.558194985348061
RUN  12 , total integrated cost =  2.5574014214686054
RUN  13 , total integrated cost =  2.556229728960253
RUN  14 , total integrated cost =  2.555421055453242
RUN  15 , total integrated cost =  2.5540627669868736
RUN  

ERROR:root:Problem in initial value trasfer


RUN  80 , total integrated cost =  2.5466424224470665
Control only changes marginally.
RUN  81 , total integrated cost =  2.5466424224470665
Improved over  81  iterations in  1.743186755105853  seconds by  96.81139581903076  percent.
Problem in initial value trasfer:  Vmean_exc -56.62446847202943 -56.62446829246389
-------  15 0.4500000000000001 0.4500000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  62.50998623209673
Gradient descend method:  None
RUN  1 , total integrated cost =  3.488399337924182
RUN  2 , total integrated cost =  3.4649882093132924
RUN  3 , total integrated cost =  3.4103792271048
RUN  4 , total integrated cost =  3.3797379044202036
RUN  5 , total integrated cost =  3.3131042020603285
RUN  6 , total integrated cost =  3.29582434858679
RUN  7 , total integrated cost =  3.2630497805059933
RUN  8 , total integrated cost =  3.253693452075768
RUN  9 , total integrated cost =  3.2396902285984805
RUN  10 , to

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  240 , total integrated cost =  3.153547528002093
Improved over  240  iterations in  4.972202444449067  seconds by  94.95513002307645  percent.
Problem in initial value trasfer:  Vmean_exc -56.67067491060143 -56.67067492413066
-------  25 0.4250000000000001 0.5000000000000002
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  19.38327335701123
Gradient descend method:  None
RUN  1 , total integrated cost =  6.222983405227982
RUN  2 , total integrated cost =  6.165865778031911
RUN  3 , total integrated cost =  5.997837621477737
RUN  4 , total integrated cost =  5.8998601362655645
RUN  5 , total integrated cost =  5.731716034621559
RUN  6 , total integrated cost =  5.6957246009309115
RUN  7 , total integrated cost =  5.651810518954451
RUN  8 , total integrated cost =  5.638414614537648
RUN  9 , total integrated cost =  5.622817275207357
RUN  10 , total integrated cost =  5.62172342318978
RUN  11 , tot

RUN  9 , total integrated cost =  1.6376891084146412
RUN  10 , total integrated cost =  1.6307044202297158
RUN  11 , total integrated cost =  1.6249973287792223
RUN  12 , total integrated cost =  1.6173181674784387
RUN  13 , total integrated cost =  1.611131412482188
RUN  14 , total integrated cost =  1.5842085850894256
RUN  15 , total integrated cost =  1.581268454517398
RUN  16 , total integrated cost =  1.5796479774013472
RUN  17 , total integrated cost =  1.5782992063067123
RUN  18 , total integrated cost =  1.5751676572719424
RUN  19 , total integrated cost =  1.5735793660413144
RUN  20 , total integrated cost =  1.573481911878551
RUN  30 , total integrated cost =  1.5690101950784563
RUN  40 , total integrated cost =  1.5657702118517447
RUN  50 , total integrated cost =  1.5652854050253155
RUN  60 , total integrated cost =  1.5648541799103861
RUN  70 , total integrated cost =  1.5646334560880497
RUN  80 , total integrated cost =  1.56403397141498
RUN  90 , total integrated cost = 

ERROR:root:Problem in initial value trasfer
ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  564 , total integrated cost =  6.206030980730793
Improved over  564  iterations in  11.583911219611764  seconds by  38.82702431091426  percent.
Problem in initial value trasfer:  Vmean_exc -56.67995817728162 -56.67995814250745
-------  95 0.5250000000000001 0.7500000000000004
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9800.648795785066
Gradient descend method:  None
RUN  1 , total integrated cost =  9800.648795785066
Control only changes marginally.
RUN  1 , total integrated cost =  9800.648795785066
Improved over  1  iterations in  0.07318508438766003  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70159251171578 -56.70154212425129
-------  115 0.4250000000000001 0.8250000000000005
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9.08061799557027
Gradient descend method:  None
RUN  1 , total integrated cost = 

RUN  80 , total integrated cost =  4.833813497667026
RUN  90 , total integrated cost =  4.8322435828094905
RUN  100 , total integrated cost =  4.830756778943847
RUN  110 , total integrated cost =  4.829321689971907
RUN  120 , total integrated cost =  4.827368023223987
RUN  130 , total integrated cost =  4.826038172180449
RUN  140 , total integrated cost =  4.824919114018363
RUN  150 , total integrated cost =  4.8241203866822575
RUN  160 , total integrated cost =  4.823125255172624
RUN  170 , total integrated cost =  4.8227065782697975
RUN  180 , total integrated cost =  4.822570545114768
RUN  190 , total integrated cost =  4.822414411984371
RUN  200 , total integrated cost =  4.822173751990256
RUN  300 , total integrated cost =  4.818696478118473
RUN  400 , total integrated cost =  4.81745687760828
RUN  500 , total integrated cost =  4.815861625260712
RUN  600 , total integrated cost =  4.815808394366489
RUN  700 , total integrated cost =  4.815748533022211
Control only changes margina

In [29]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        counter += 1

ERROR:root:Problem in initial value trasfer


--------------- 0
[[True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [True, True], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False], [True, True], [False, False]]
-------  5 0.4000000000000001 0.40000000000000013
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.5466424224470665
Gradient descend method:  None
RUN  1 , total integrated cost =  2.5466424224470665
Control only changes marginally.
RUN  1 , total integrated cost =  2.5466424224470665
Improved over  1  iterations in  0.10804298147559166  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.62446847202943 -56.62446829246389
-------  15 0.4500

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  3.153547528002093
Control only changes marginally.
RUN  1 , total integrated cost =  3.153547528002093
Improved over  1  iterations in  0.11170490644872189  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67067491060143 -56.67067492413066
-------  25 0.4250000000000001 0.5000000000000002
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5.5882863709142665
Gradient descend method:  None
RUN  1 , total integrated cost =  5.5882863709142665
Control only changes marginally.
RUN  1 , total integrated cost =  5.5882863709142665
Improved over  1  iterations in  0.11063752509653568  seconds by  0.0  percent.
-------  45 0.5000000000000002 0.5750000000000003
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  3.834391526161677
Gradient descend method:  None
RUN  1 , total integrated cost =  3.834391526161

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  1.563033938450464
Control only changes marginally.
RUN  1 , total integrated cost =  1.563033938450464
Improved over  1  iterations in  0.11471140943467617  seconds by  0.0  percent.
-------  85 0.47500000000000014 0.7250000000000004
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  6.206030980730793
Gradient descend method:  None
RUN  1 , total integrated cost =  6.206030980730793
Control only changes marginally.
RUN  1 , total integrated cost =  6.206030980730793
Improved over  1  iterations in  0.11394384875893593  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.67995817728162 -56.67995814250745
-------  95 0.5250000000000001 0.7500000000000004
converged for  95
-------  115 0.4250000000000001 0.8250000000000005
no convergence
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8.55731265982517
Gradient desc

In [30]:
print(conv_1[::i_stepsize])

with open(final_file_1,'wb') as f:
    pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)

[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
